<a href="https://colab.research.google.com/github/safnanazar04-cmyk/intermediate_assessment2/blob/main/Intermediate_assessment2_Safna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report

In [ ]:
train = pd.read_csv('/content/drive/MyDrive/DSA Activities /Case Study/train_LZdllcl.csv')
test = pd.read_csv('/content/drive/MyDrive/DSA Activities /Case Study/test_2umaH9m.csv')

In [ ]:
print("Train shape:",train.shape)
print("Test Shape:",test.shape)

Train shape: (54808, 14)
Test Shape: (23490, 13)


In [ ]:
print("Train columns:")
train.columns

Train columns:


Index(['employee_id', 'department', 'region', 'education', 'gender',
       'recruitment_channel', 'no_of_trainings', 'age', 'previous_year_rating',
       'length_of_service', 'KPIs_met >80%', 'awards_won?',
       'avg_training_score', 'is_promoted'],
      dtype='object')

In [ ]:
print("Test columns:")
test.columns

Test columns:


Index(['employee_id', 'department', 'region', 'education', 'gender',
       'recruitment_channel', 'no_of_trainings', 'age', 'previous_year_rating',
       'length_of_service', 'KPIs_met >80%', 'awards_won?',
       'avg_training_score'],
      dtype='object')

EDA -- train

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54808 entries, 0 to 54807
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   employee_id           54808 non-null  int64  
 1   department            54808 non-null  object 
 2   region                54808 non-null  object 
 3   education             52399 non-null  object 
 4   gender                54808 non-null  object 
 5   recruitment_channel   54808 non-null  object 
 6   no_of_trainings       54808 non-null  int64  
 7   age                   54808 non-null  int64  
 8   previous_year_rating  50684 non-null  float64
 9   length_of_service     54808 non-null  int64  
 10  KPIs_met >80%         54808 non-null  int64  
 11  awards_won?           54808 non-null  int64  
 12  avg_training_score    54808 non-null  int64  
 13  is_promoted           54808 non-null  int64  
dtypes: float64(1), int64(8), object(5)
memory usage: 5.9+ MB


In [ ]:
train.isnull().sum()

,0
employee_id,0
department,0
region,0
education,2409
gender,0
recruitment_channel,0
no_of_trainings,0
age,0
previous_year_rating,4124
length_of_service,0


In [ ]:
train.duplicated().sum()

np.int64(0)

In [ ]:
train["is_promoted"].value_counts()

,count
is_promoted,
0,50140
1,4668


In [ ]:
train["is_promoted"].value_counts(normalize=True)*100

,proportion
is_promoted,
0,91.482995
1,8.517005


EDA --test

In [ ]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23490 entries, 0 to 23489
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   employee_id           23490 non-null  int64  
 1   department            23490 non-null  object 
 2   region                23490 non-null  object 
 3   education             22456 non-null  object 
 4   gender                23490 non-null  object 
 5   recruitment_channel   23490 non-null  object 
 6   no_of_trainings       23490 non-null  int64  
 7   age                   23490 non-null  int64  
 8   previous_year_rating  21678 non-null  float64
 9   length_of_service     23490 non-null  int64  
 10  KPIs_met >80%         23490 non-null  int64  
 11  awards_won?           23490 non-null  int64  
 12  avg_training_score    23490 non-null  int64  
dtypes: float64(1), int64(7), object(5)
memory usage: 2.3+ MB


In [ ]:
test.isnull().sum()

,0
employee_id,0
department,0
region,0
education,1034
gender,0
recruitment_channel,0
no_of_trainings,0
age,0
previous_year_rating,1812
length_of_service,0


In [ ]:
test.duplicated().sum()

np.int64(0)

In [ ]:
X = train.drop(columns=["is_promoted"],axis=1)
y = train["is_promoted"]

In [ ]:
X_test = test.copy()

In [ ]:
numerical_features = X.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_columns = X.select_dtypes(include="object").columns.tolist()

In [ ]:
print("Numerical Features:")
print(numerical_features)
print("Categorical Features:")
print(categorical_columns)

Numerical Features:
['employee_id', 'no_of_trainings', 'age', 'previous_year_rating', 'length_of_service', 'KPIs_met >80%', 'awards_won?', 'avg_training_score']
Categorical Features:
['department', 'region', 'education', 'gender', 'recruitment_channel']


In [ ]:
X_train,X_valid,y_train,y_valid = train_test_split(X,y,test_size= 0.20,random_state=42,stratify=y)
print("Training Data:",X_train.shape)
print("Validation Data:",X_valid.shape)

Training Data: (43846, 13)
Validation Data: (10962, 13)


In [ ]:
numerical_transformer = Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

In [ ]:
categorical_transformer = Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
])

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num",numerical_transformer,numerical_features),
    ("cat",categorical_transformer,categorical_columns)
])

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test)

print("Training Data:",X_train_processed.shape)
print("Validation Data:",X_valid_processed.shape)
print("Test Data:",X_test_processed.shape)

Training Data: (43846, 59)
Validation Data: (10962, 59)
Test Data: (23490, 59)


In [ ]:
model = RandomForestRegressor(n_estimators=200,random_state=42,n_jobs=-1)
model.fit(X_train_processed,y_train)

RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
y_pred = model.predict(X_valid_processed)
# Convert continuous predictions to binary using a threshold (e.g., 0.5)
y_pred_binary = (y_pred > 0.5).astype(int)

accuracy = accuracy_score(y_valid,y_pred_binary)
print("Accuracy:",accuracy)
print("Classification Report")
print(classification_report(y_valid,y_pred_binary))

Accuracy: 0.9405218025907681
Classification Report
              precision    recall  f1-score   support

           0       0.94      1.00      0.97     10028
           1       0.87      0.35      0.50       934

    accuracy                           0.94     10962
   macro avg       0.91      0.67      0.74     10962
weighted avg       0.94      0.94      0.93     10962



In [ ]:
X_processed = preprocessor.fit_transform(X)
X_test_processed = preprocessor.transform(X_test)
final_model = RandomForestClassifier(n_estimators=300,random_state=42,n_jobs=-1)
final_model.fit(X_processed,y)

RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)

In [ ]:
test_predictions = final_model.predict(X_test_processed)
print("Number of Test Predictions:",len(test_predictions))

Number of Test Predictions: 23490


In [ ]:
sample_submission = pd.read_csv("/content/drive/MyDrive/DSA Activities /Case Study/sample_submission_M0L0uXE.csv")

In [ ]:
submission = sample_submission.copy()
print("Sample submission columns:")
print(submission.columns)

submission["is_promoted"] = test_predictions

Sample submission columns:
Index(['employee_id', 'is_promoted'], dtype='object')


In [ ]:
submission.to_csv("Final_submission.csv",index=False)
print("Submission file created successfully!")
print("File name: final_submission.csv")
print("First 10 predictions:")
print(submission.head(10))

Submission file created successfully!
File name: final_submission.csv
First 10 predictions:
   employee_id  is_promoted
0         8724            0
1        74430            0
2        72255            0
3        38562            0
4        64486            0
5        46232            0
6        54542            0
7        67269            0
8        66174            0
9        76303            0
